In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


In [2]:
# ----------------------------------------------------------
# 1. Load Dataset
# ----------------------------------------------------------
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country",
    "income"
]

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    names=columns,
    sep=",",
    skipinitialspace=True
)

In [3]:
# ----------------------------------------------------------
# 2. Handle Missing Values
# ----------------------------------------------------------
df = df.replace("?", np.nan)
df = df.dropna()

# ----------------------------------------------------------
# 3. Encode Target Variable
# ----------------------------------------------------------
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# ----------------------------------------------------------
# 4. Separate Features and Target
# ----------------------------------------------------------
X = df.drop("income", axis=1)
y = df["income"].values

# ----------------------------------------------------------
# 5. One-Hot Encode Categorical Features
# ----------------------------------------------------------
X = pd.get_dummies(X)

# ----------------------------------------------------------
# 6. Train–Test Split
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------------------------------------
# 7. Feature Scaling
# ----------------------------------------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [4]:
# ----------------------------------------------------------
# 8. Convert to PyTorch Tensors
# ----------------------------------------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [69]:
# ----------------------------------------------------------
# 10. Define MLP Model (2 Hidden Layers)
# ----------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = MLP(X_train.shape[1])

In [70]:
# ----------------------------------------------------------
# 11. Loss Function & Optimizer
# ----------------------------------------------------------
criterion= nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [71]:
# ----------------------------------------------------------
# 12. Training Loop
# ----------------------------------------------------------
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = criterion(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.7439
Epoch 5, Loss: 0.4066
Epoch 10, Loss: 0.3713
Epoch 15, Loss: 0.3496
Epoch 20, Loss: 0.3357
Epoch 25, Loss: 0.3252
Epoch 30, Loss: 0.3194
Epoch 35, Loss: 0.3132
Epoch 40, Loss: 0.3066
Epoch 45, Loss: 0.2996


In [72]:
# ----------------------------------------------------------
# 13. Evaluation
# ----------------------------------------------------------
with torch.no_grad():
    y_prob = model(X_test)
    y_pred = (y_prob >= 0.35).int()

y_test_np = y_test.numpy()
y_pred_np = y_pred.numpy()

In [74]:
# ----------------------------------------------------------
# 14. Metrics Reporting
# ----------------------------------------------------------
accuracy = accuracy_score(y_test_np, y_pred_np)
precision = precision_score(y_test_np, y_pred_np)
recall = recall_score(y_test_np, y_pred_np)
f1 = f1_score(y_test_np, y_pred_np)

print("\nEvaluation Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")




Evaluation Metrics:
Accuracy : 0.8369
Precision: 0.6502
Recall   : 0.7463
F1-score : 0.6950
